In [5]:
import sys
!{sys.executable} -m pip install --no-cache-dir -U "requests>=2.31,<3" "urllib3>=2.6,<3"

  Attempting uninstall: urllib3
    Found existing installation: urllib3 1.25.11
    Uninstalling urllib3-1.25.11:
      Successfully uninstalled urllib3-1.25.11
  Attempting uninstall: requests
    Found existing installation: requests 2.23.0
    Uninstalling requests-2.23.0:
      Successfully uninstalled requests-2.23.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [requests]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
app-store-scraper 0.3.5 requires requests==2.23.0, but you have requests 2.32.5 which is incompatible.


In [8]:
import requests
import json
import time

app_id = 1374540454
country = "ru"

all_reviews = []
seen_ids = set()

for page in range(1, 11):
    url = f"https://itunes.apple.com/{country}/rss/customerreviews/page={page}/id={app_id}/sortby=mostrecent/json"
    r = requests.get(url, timeout=30)

    if r.status_code != 200:
        print("Stop on page", page, "status", r.status_code)
        break

    data = r.json()
    entries = data.get("feed", {}).get("entry", [])

    if not entries:
        print("No entries on page", page)
        break

    # если вдруг Apple вернула один объект, а не список
    if isinstance(entries, dict):
        entries = [entries]

    page_added = 0

    for entry in entries:
        if not isinstance(entry, dict):
            continue

        # пропускаем карточку самого приложения
        if "im:rating" not in entry:
            continue

        entry_id = entry.get("id")
        if isinstance(entry_id, dict):
            review_id = entry_id.get("label")
        else:
            review_id = str(entry_id)

        if not review_id or review_id in seen_ids:
            continue
        seen_ids.add(review_id)

        author = entry.get("author", {})
        if isinstance(author, dict):
            author_name = author.get("name", {})
            if isinstance(author_name, dict):
                author_name = author_name.get("label")
        else:
            author_name = None

        title = entry.get("title")
        if isinstance(title, dict):
            title = title.get("label")

        content = entry.get("content")
        if isinstance(content, dict):
            content = content.get("label")

        rating = entry.get("im:rating")
        if isinstance(rating, dict):
            rating = rating.get("label")
        try:
            rating = int(rating)
        except:
            rating = None

        version = entry.get("im:version")
        if isinstance(version, dict):
            version = version.get("label")

        updated = entry.get("updated")
        if isinstance(updated, dict):
            updated = updated.get("label")

        all_reviews.append({
            "review_id": review_id,
            "author": author_name,
            "title": title,
            "content": content,
            "rating": rating,
            "version": version,
            "updated": updated,
        })
        page_added += 1

    print(f"Page {page}: +{page_added} reviews")
    time.sleep(1)

with open("appstore_reviews_rss.json", "w", encoding="utf-8") as f:
    json.dump(all_reviews, f, ensure_ascii=False, indent=2)

print("Total reviews:", len(all_reviews))

Page 1: +50 reviews
Page 2: +50 reviews
Page 3: +50 reviews
Page 4: +50 reviews
Page 5: +50 reviews
Page 6: +50 reviews
Page 7: +50 reviews
Page 8: +50 reviews
Page 9: +1 reviews
No entries on page 10
Total reviews: 401
